# FR3 Duo handover · SmolVLA training on free Colab

Pipeline R1 → R4 on a free Colab GPU, then the model + model card to a **private** Hugging Face model repo.

**Rules**
- The dataset is **never** uploaded to the Hub. It comes from *your* Google Drive (organiser data, team use only).
- Registry and outputs live on Drive, so they survive a disconnect. Checkpoints too → training can resume.
- A **human** approves the dataset (step 6). Numbers only from logged runs.

**Before the first run** — put the two datasets in `MyDrive/shs-hack-2026/data/` (from the organisers' file share),
either as the zips (`fr3_duo_handover_84_ep.zip`, `FR3Duo_Handover_Dataset_lerobot3.zip`) or as the extracted
folders (each with `meta/`, `data/`, `videos/` inside). Step 4 finds them either way.
Add a Colab secret `HF_TOKEN` (🔑 icon, write access) — only needed for step 10.

**Runtime → Change runtime type → T4 GPU.**

## 0 · Settings

In [ ]:
REPO_URL = "https://github.com/RohithThota97/shs-hack-2026-vla-pipeline.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/shs-hack-2026"       # data zips in DRIVE_DIR/data, state in DRIVE_DIR/state
DATASETS = {          # data/raw/<name> ← (episode count, zip name) — found in DRIVE_DIR/data as folder or zip
    "fr3_duo_handover_84": (84, "fr3_duo_handover_84_ep.zip"),
    "fr3_duo_handover_lighting10": (10, "FR3Duo_Handover_Dataset_lerobot3.zip"),
}
LEROBOT_SPEC = "lerobot[smolvla,dataset]==0.6.1"   # datasets were written with 0.6.1; [dataset] pins av<16 + torchcodec
TRAIN_CONFIG = "configs/train/smolvla_fr3_duo.yaml"
HF_MODEL_REPO = "RohithThota/smolvla-fr3duo-handover"   # private model repo
REPO = "/content/repo"

## 1 · GPU, Drive, code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/state/registry", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/state/outputs/logs", exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
%cd {REPO}
!git log --oneline -1

## 2 · Install

LeRobot may install a different torch than Colab ships. If the output says to **restart the session**, do it
(Runtime → Restart session) and re-run cells 0 and 1, then continue at step 3.

In [ ]:
!pip install -q -e ".[dev,video,hub]" "{LEROBOT_SPEC}" matplotlib
!pip freeze | grep -i -E "^(lerobot|torch|torchvision|torchcodec|av|huggingface.hub)=="
import av; assert av.__version__.startswith("15."), f"PyAV {av.__version__}: LeRobot 0.6.1 needs av>=15,<16 — restart the session"

## 3 · Persistent state on Drive

`registry/` and `outputs/` in the repo become links to Drive. Demo IDs depend on `data/raw/<name>`, so they are
the same on every Colab session (and on a laptop with the same layout).

In [ ]:
import shutil
for name in ("registry", "outputs"):
    p = f"{REPO}/{name}"
    if os.path.islink(p):
        continue
    shutil.rmtree(p, ignore_errors=True)
    os.symlink(f"{DRIVE_DIR}/state/{name}", p)
!ls -la registry outputs

## 4 · Data: Drive → local disk (fast), linked into `data/raw/`

Uses extracted folders on Drive if present (told apart by episode count), else unzips the zips. A copy is only
reused if it finished (`.complete` marker). If the cell can't decide, set `SRC` by hand to the folder paths it prints.

In [ ]:
import glob, json, shutil, zipfile
SRC = {}   # optional, by hand: {"fr3_duo_handover_84": "/content/drive/MyDrive/.../fr3_duo_handover_v9_night_84", ...}
if not SRC:
    by_eps = {}
    for f in glob.glob(f"{DRIVE_DIR}/data/**/meta/info.json", recursive=True):
        root = os.path.dirname(os.path.dirname(f))
        n = json.load(open(f))["total_episodes"]
        print(f"found folder: {n} episodes ← {root}")
        by_eps.setdefault(n, []).append(root)
    for name, (n_eps, _) in DATASETS.items():
        if len(by_eps.get(n_eps, [])) > 1:
            raise RuntimeError(f"{len(by_eps[n_eps])} folders with {n_eps} episodes — set SRC by hand")
        if by_eps.get(n_eps):
            SRC[name] = by_eps[n_eps][0]

os.makedirs(f"{REPO}/data/raw", exist_ok=True)
for name, (n_eps, zip_name) in DATASETS.items():
    link, dest = f"{REPO}/data/raw/{name}", f"/content/data_cache/{name}"
    if not os.path.exists(f"{dest}/.complete"):
        shutil.rmtree(dest, ignore_errors=True)               # half-finished copy from a dropped session
        if name in SRC:
            print(f"copying {name} from Drive …")
            shutil.copytree(SRC[name], dest)
        else:
            z = f"{DRIVE_DIR}/data/{zip_name}"
            assert os.path.exists(z), f"{name}: no folder with {n_eps} episodes and no {zip_name} under {DRIVE_DIR}/data"
            print(f"unzipping {zip_name} …")
            zipfile.ZipFile(z).extractall(dest)
        open(f"{dest}/.complete", "w").close()
    info = sorted(glob.glob(f"{dest}/**/meta/info.json", recursive=True), key=len)[0]
    root = os.path.dirname(os.path.dirname(info))
    if os.path.islink(link): os.unlink(link)
    os.symlink(root, link)
    files = sum(len(fs) for _, _, fs in os.walk(root))
    print(f"ready {name}: {files} files ← {SRC.get(name, zip_name)}")
!python -c "import json;[print(n, {k: json.load(open(f'data/raw/{n}/meta/info.json'))[k] for k in ('total_episodes','total_frames','fps')}) for n in ('fr3_duo_handover_84','fr3_duo_handover_lighting10')]"

## 5 · AV1 decode check (likely blocker)

All videos are AV1. This loads samples through LeRobot itself — the same path training uses — and times reads.
If it fails, try `VIDEO_BACKEND = "pyav"`; if that fails too, use the re-encode cell below (creates a new dataset
version, because the video files change).

In [ ]:
import time, random
from lerobot.datasets.lerobot_dataset import LeRobotDataset   # path differs in some LeRobot versions
VIDEO_BACKEND = None    # None = LeRobot default; or "pyav" / "torchcodec"
kw = {"video_backend": VIDEO_BACKEND} if VIDEO_BACKEND else {}
ds = LeRobotDataset("local/fr3_duo_handover_84", root=f"{REPO}/data/raw/fr3_duo_handover_84", **kw)
x = ds[0]
print({k: tuple(v.shape) for k, v in x.items() if hasattr(v, "shape")})
t0 = time.time()
for i in random.sample(range(len(ds)), 100):
    ds[i]
print(f"{100 / (time.time() - t0):.1f} samples/s (single process) — training needs ≥ batch_size / s_per_step")

In [ ]:
# ONLY if decoding failed: re-encode the videos to h264 into a new folder, then point data/raw at it.
# !lerobot-edit-dataset --repo_id local/fr3_duo_handover_84 --root data/raw/fr3_duo_handover_84 \
#     --new_root /content/data_cache/fr3_duo_handover_84_h264 \
#     --operation.type reencode_videos --operation.rgb_encoder.vcodec h264

## 6 · R1–R3: ingest, quality check, dataset version

In [ ]:
!pytest -q 2>&1 | tail -2
for name in ("fr3_duo_handover_84", "fr3_duo_handover_lighting10"):
    !hackpipe ingest --root data/raw/{name}
    !hackpipe qc --root data/raw/{name} | head -20
!hackpipe dataset list

In [ ]:
# Creates the next draft version (ds-v1 on the first run). Skip if `dataset list` already shows the one you need.
!hackpipe dataset create --notes "v1: night84 90/10 train/val; test = lighting10 (held-out source)"

In [ ]:
DS = "ds-v1"     # the version printed above
from IPython.display import Markdown, display
display(Markdown(open(f"outputs/qc/fr3_duo_handover_84/qc_report.md").read()))
display(Markdown(open(f"outputs/datasets/{DS}/dataset_card.md").read()))

### ⛔ STOP — human approval

Read the QC report and the dataset card above. If they are right, put your name below and run the cell.
Agents never run this cell.

In [ ]:
APPROVED_BY = ""        # your name
assert APPROVED_BY, "A human must review and approve the dataset"
!hackpipe dataset approve {DS}
!hackpipe status

## 7 · Dry run — check the `lerobot-train` command line by line

In [ ]:
!hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --dry-run

## 8 · Timing run (~15 min) → seconds/step and peak VRAM

The timing run gets a model ID too: note it as "timing run, not a result". Out of memory → set
`EXTRA = "--set batch_size=4"` and repeat (and use the same batch size for the full run).

In [ ]:
EXTRA = ""
import threading, time
vram, stop = [], threading.Event()
def poll():
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip()
        if out: vram.append(int(out.splitlines()[0]))
        time.sleep(5)
th = threading.Thread(target=poll, daemon=True); th.start()
!hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --set steps=200 --set save_freq=200 {EXTRA} 2>&1 | tee outputs/logs/train_timing.log
stop.set()
print("peak VRAM MiB:", max(vram) if vram else "n/a")
!grep -E "step:" outputs/logs/train_timing.log | tail -5

## 9 · Full training run

Set the step budget from the timing run: `steps = slot_seconds / seconds_per_step`, rounded down to a multiple of
`save_freq` (1000). Free Colab sessions end after a few hours — checkpoints every 1000 steps are on Drive, and the
model is registered when training starts, so a dropped session can resume.

In [ ]:
SEC_PER_STEP = None     # from the timing log (updt_s + data_s per step)
SLOT_HOURS = 3.5        # what you expect Colab to give you in one session
STEPS = int(SLOT_HOURS * 3600 / SEC_PER_STEP) // 1000 * 1000
print("steps:", STEPS, "→ record the reason in docs/decisions.md and commit the value in", TRAIN_CONFIG)

In [ ]:
!hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --set steps={STEPS} {EXTRA} 2>&1 | tee -a outputs/logs/train_v1.log

In [ ]:
# After a disconnect: re-run cells 0–4, then resume with the model ID printed when training started.
# !hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --resume <model_id> 2>&1 | tee -a outputs/logs/train_v1.log

In [ ]:
MODEL = ""     # the model ID of the full run (printed when training started)
!python scripts/plot_loss.py --log outputs/logs/train_v1.log --model {MODEL}
!hackpipe status
from IPython.display import Image
Image(f"outputs/runs/{MODEL}/loss.png")

## 10 · Model + model card → Hugging Face (private)

Uploads only the checkpoint, `README.md` (model card), run record, dataset summary (IDs and counts, no data)
and the loss curve. Check the dry run first.

In [ ]:
from google.colab import userdata
from IPython.display import Markdown, display
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert HF_MODEL_REPO, "Set HF_MODEL_REPO in step 0"
!hackpipe publish --model {MODEL} --repo {HF_MODEL_REPO} --dry-run
display(Markdown(open(f"outputs/hub/{MODEL}/README.md").read()))

In [ ]:
!hackpipe publish --model {MODEL} --repo {HF_MODEL_REPO}